# Using Tools

## What are tools?
* Some use cases require Models to interface directly with tools: external systems—such as APIs, databases, or file systems—using structured input.
* Some Models (e.g., OpenAI, Anthropic, and Gemini) feature built-in tools that are executed server-side, such as web search and code interpreters. See the [integrations](https://docs.langchain.com/oss/python/integrations) page for more info about it.
* You can also create custom tools.
* Tools extend the Model capabilities by letting it interact with the world through well-defined inputs and outputs.
* Agentic LLM Applications (aka Agents) can decide if they need to use an available tool to achieve a goal.

## As always, let's load the .env file first
* Remember, in the .env file we have our OpenAI API key and the Tavily API key (this will allow us to use the tool later).

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

#### Remember how you can get your Tavily API key

Here's how to get your Tavily API key in simple steps:

**1. Go to Tavily's website**
Visit [tavily.com](https://tavily.com) and click on "Get API Key" or "Sign Up"

**2. Create an account**
Sign up using your email address or through a social login (Google, GitHub, etc.)

**3. Verify your email**
Check your inbox for a verification email and click the link to confirm your account

**4. Get your free API key**
Once logged in, you'll see your API key on the dashboard. Tavily offers a free tier that includes 1,000 API calls per month, which is more than enough for learning and doing course exercises

**5. Copy and save the key**
Copy the API key (it looks like a long string of letters and numbers). Store it somewhere safe - you'll need it for your LangChain exercises

**6. Add it in your .env file**
```
TAVILY_API_KEY=your-api-key-here
```

**Important tip:** Never share your API key publicly or commit it to GitHub. The free tier should be sufficient for most course exercises, but you can always check your usage on the Tavily dashboard.

## First, let's try an "Agent" without tools

#### Update note: current OpenAI models
* The original examples are very good for learning the concepts, and they worked with the OpenAI models available when the course was recorded: they used `gpt-4o-mini`.
* OpenAI has since announced the retirement of `gpt-4o-mini` (shutdown on January 14, 2027) and recommends `gpt-6-luna` as its replacement. With the current models, we would do it like this: `MODEL = "gpt-6-luna"`.
* The agents with tools below need one more detail. Through the endpoint LangChain uses by default, `gpt-6-luna` only accepts tools when reasoning is turned off. That is why we create the model with `init_chat_model(MODEL, reasoning_effort="none")` and pass it to those agents.
* We removed the date from the question, so you always get today's weather.
* When the agent uses the web search, the answer may end with a strange text like `citeturn6f4581`. It is a citation marker: recent OpenAI models mark the source of each piece of information. ChatGPT turns these markers into links; through the API they arrive raw. That is why we remove them with a small function before printing the answer. See [OpenAI's citation formatting guide](https://developers.openai.com/api/docs/guides/citation-formatting).
* The explanations below still show `gpt-4o-mini`: the ideas are the same.
* Check the current models at [developers.openai.com/api/docs/models](https://developers.openai.com/api/docs/models).

In [ ]:
from langchain.agents import create_agent

# Original version: this example used gpt-4o-mini.
# OpenAI has deprecated it (shutdown on January 14, 2027) and recommends
# gpt-6-luna as its replacement: small, fast and affordable for practice.
MODEL = "gpt-6-luna"

system_prompt = "You are a weather forecast assistant."

agent = create_agent(
    model=MODEL,
    system_prompt=system_prompt
)

from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [HumanMessage(content="How is the weather today in San Francisco?")]}
)

print(response['messages'][-1].content)

## Then, let's try a very basic Agent using one external online search tool (Tavily)

In [ ]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:

    """Search the web for information"""

    return tavily_client.search(query)

#testing the tool
web_search.invoke("How is the weather today in San Francisco?")

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model

# This agent uses tools. Through the Chat Completions endpoint, which
# LangChain uses by default, gpt-6-luna only accepts tools when reasoning
# is turned off, so we create the model with reasoning_effort="none".
# The model then answers directly, like gpt-4o-mini did.
model = init_chat_model(MODEL, reasoning_effort="none")

weather_agent = create_agent(
    model=model,
    tools=[web_search]
)

In [ ]:
from langchain.messages import HumanMessage

question = HumanMessage(content="How is the weather today in San Francisco?")

response = weather_agent.invoke(
    {"messages": [question]}
)

# Recent OpenAI models mark the sources they use with citation markers,
# for example "cite turn6f4581" wrapped in invisible characters.
# Apps like ChatGPT turn them into links. Here we just remove them,
# as OpenAI's citation formatting guide suggests when you don't show links.
import re

def remove_citations(text):
    """Remove OpenAI citation markers from the model's answer."""
    return re.sub(r"\ue200cite\ue202.*?\ue201", "", text).strip()

print(remove_citations(response['messages'][-1].content))

In [ ]:
from pprint import pprint

pprint(response['messages'])

In [ ]:
print(response["messages"][1].tool_calls)

## Let's explain the previous code in simple terms

Below is the same code, explained **in simple terms, line by line**, as if you’re new to Python + LangChain.

---

#### 1) Load environment variables (API keys, secrets)

```python
from dotenv import load_dotenv
```

* Imports a helper function from the `python-dotenv` package.
* This package lets you store secrets (like API keys) in a `.env` file.

```python
load_dotenv()
```

* Reads your `.env` file and loads its values into “environment variables”.
* Example: if your `.env` contains `TAVILY_API_KEY=...`, your program can now access it automatically.

---

#### 2) Imports for making a LangChain tool + typing

```python
from langchain.tools import tool
```

* Imports the `@tool` decorator.
* A “tool” in LangChain is basically a normal Python function that the agent is allowed to call.

```python
from typing import Dict, Any
```

* Imports type hints:

  * `Dict[str, Any]` means “a dictionary with string keys and any type of values”.
* Type hints help humans (and editors) understand what the function returns.

```python
from tavily import TavilyClient
```

* Imports Tavily’s Python client.
* Tavily is a web search API service.

---

#### 3) Create a Tavily client (the object that performs searches)

```python
tavily_client = TavilyClient()
```

* Creates a Tavily client instance.
* Behind the scenes, this client reads your Tavily API key from the environment variables (which is why you loaded `.env` earlier).

---

#### 4) Define a LangChain tool function: `web_search`

```python
@tool
def web_search(query: str) -> Dict[str, Any]:
```

* `@tool` turns this Python function into a **LangChain tool**.
* The agent can call it later.
* The function takes one input:

  * `query: str` → a search string
* It returns:

  * `Dict[str, Any]` → search results in dictionary form

```python
    """Search the web for information"""
```

* This is the function’s docstring (a description).
* Agents can sometimes read this description to decide when to use the tool.

```python
    return tavily_client.search(query)
```

* Actually performs a web search using Tavily.
* Whatever Tavily returns (usually a dict with results) is returned by your tool.

---

## 5) Test the tool directly (without an agent)

```python
#testing the tool
web_search.invoke("How is the weather today in San Francisco?")
```

* This calls the tool **manually**.
* `invoke(...)` is LangChain’s standard way to run a tool.
* This is just to confirm “the tool works” before giving it to an agent.

---

## 6) Create an agent and give it your tool

```python
from langchain.agents import create_agent
```

* Imports a helper that builds an “agent”.
* An **agent** is an LLM app that can decide:

  * “I should answer directly” OR
  * “I should call a tool to get information”

```python
weather_agent = create_agent(
    model="gpt-4o-mini",
    tools=[web_search]
)
```

* Creates an agent named `weather_agent`.
* `model="gpt-4o-mini"`: chooses which LLM to use.
* `tools=[web_search]`: gives the agent permission to call your `web_search` tool.
* So the agent can do things like: “To answer how is the weather today in a particular location, I should search the web.” (Remember: as we saw with the previous Agent, the LLM cannot answer this question directly since it was trained with past data, not today's data. That is why it will need Tavily for that).

---

## 7) Create a user message

```python
from langchain.messages import HumanMessage
```

* Imports the “message object” representing something a human/user says.

```python
question = HumanMessage(content="How is the weather today in San Francisco?")
```

* Builds a human message with your question inside it.
* Agents often work with chat-like “messages” rather than plain strings.

---

## 8) Ask the agent the question (agent may call tools)

```python
response = weather_agent.invoke(
    {"messages": [question]}
)
```

* Runs the agent.
* You pass a dictionary with `"messages"` containing the conversation so far.
* The agent will:

  1. Read the question
  2. Decide if it needs the web
  3. Potentially call `web_search(...)`
  4. Produce a final answer message

The result (`response`) is typically a dictionary containing a `"messages"` list (the conversation including tool calls + final answer).

---

## 9) Print the final answer text

```python
print(response['messages'][-1].content)
```

* `response['messages']` is a list of messages.
* `[-1]` means “the last message”.
* `.content` is the text of that message.
* So this prints the agent’s final answer.

---

## 10) Pretty-print the full message history

```python
from pprint import pprint
```

* Imports a nicer printing function for complex Python objects.

```python
pprint(response['messages'])
```

* Prints *all* messages nicely.
* This often includes:

  * your HumanMessage
  * tool call request(s)
  * tool output message(s)
  * final AI answer

---

## 11) Inspect the tool calls the agent made

```python
print(response["messages"][1].tool_calls)
```

* Looks at the **second message** in the list (`[1]`).
* Often, message `[1]` is the agent message where it decided to call tools (depends on the agent/framework behavior).
* `.tool_calls` shows what tool(s) it tried to call, including arguments (like the query it used).

---

# Mental model summary (super simple)

* You created a **web search tool** (`web_search`) that uses Tavily.
* You created an **agent** that can use that tool.
* You asked: “What’s the weather?”
* The agent can decide: “I need the web → call `web_search` → read results → answer.”

## How to run this code from Visual Studio Code
* Open Terminal.
* Make sure you are in the project folder.
* Make sure you have the poetry env activated.
* Enter and run the following command:
    * `python 006-tools.py` 

## Advanced Tip: Tools are most powerful when they can access agent state, runtime context, and long-term memory.
This enables tools to make context-aware decisions, personalize responses, and maintain information across conversations.

Tools can access runtime information through the `ToolRuntime` parameter, which provides:
* State - Mutable data that flows through execution (e.g., messages, counters, custom fields)
* Context - Immutable configuration like user IDs, session details, or application-specific configuration
* Store - Persistent long-term memory across conversations
* Stream Writer - Stream custom updates as tools execute
* Config - RunnableConfig for the execution
* Tool Call ID - ID of the current tool call

## The `ToolRuntime` parameter
Let us explain the `ToolRuntime` parameter in simple terms.

#### What is `ToolRuntime`?

Think of `ToolRuntime` as a **magic backpack** that your tool automatically gets access to when it runs. This backpack contains important information and capabilities that your tool might need.


#### What's Inside the Backpack?

`ToolRuntime` gives your tool access to 6 important things:

1. **State** - Information that changes as the conversation progresses (like the messages in the chat)
2. **Context** - Fixed settings (like who the user is)
3. **Store** - Long-term memory (like a database to remember things between conversations)
4. **Streaming** - Ability to send updates while the tool is working
5. **Config** - Technical settings
6. **Tool Call ID** - A unique identifier for this specific tool run


#### Why is This Useful?

Before `ToolRuntime`, you had to use 4 different complicated methods to access these things. Now, you just need **one simple parameter** - much easier'. `ToolRuntime` replaces the older pattern of using separate
    * InjectedState,
    * InjectedStore,
    * get_runtime,
    * and InjectedToolCallId annotations.

#### Example 1: Counting Messages

```python
from langchain.tools import tool, ToolRuntime

# Access the current conversation state
@tool
def summarize_conversation(
    runtime: ToolRuntime
) -> str:
    """Summarize the conversation so far."""
    messages = runtime.state["messages"]

    human_msgs = sum(1 for m in messages if m.__class__.__name__ == "HumanMessage")
    ai_msgs = sum(1 for m in messages if m.__class__.__name__ == "AIMessage")
    tool_msgs = sum(1 for m in messages if m.__class__.__name__ == "ToolMessage")

    return f"Conversation has {human_msgs} user messages, {ai_msgs} AI responses, and {tool_msgs} tool results"
```

**What's happening here:**
- The tool receives `runtime` automatically (you don't pass it manually)
- It looks inside `runtime.state` to find all the messages in the conversation
- It counts how many messages came from the human, AI, and tools
- Returns a summary like "Conversation has 3 user messages, 3 AI responses, and 2 tool results"


#### Example 2: Getting User Preferences

```python
# Access custom state fields
@tool
def get_user_preference(
    pref_name: str,
    runtime: ToolRuntime  # ToolRuntime parameter is not visible to the model
) -> str:
    """Get a user preference value."""
    preferences = runtime.state.get("user_preferences", {})
    return preferences.get(pref_name, "Not set")
```

**What's happening here:**
- This tool takes TWO parameters: `pref_name` (what preference to look up) and `runtime`
- **Important**: The AI model only "sees" `pref_name` - it doesn't know about `runtime`
- `runtime` is automatically provided by the system behind the scenes
- The tool looks in the conversation's state for user preferences
- If the preference exists, it returns the value; otherwise returns "Not set"


#### Key Takeaway

`ToolRuntime` is like having a **special helper** that gives your tool access to all the context it needs without cluttering your tool's visible parameters. It's automatically provided, the AI doesn't see it, and it makes your tools much more powerful.

## Basic Example: using ToolRuntime to access Custom Context

In [ ]:
from dataclasses import dataclass

# Here is where we create the custom context
@dataclass
class Preferences:
    vehicle: str = "Vespa"
    city: str = "San Francisco"

from langchain.tools import tool, ToolRuntime

# Here is where we use the custom context
@tool
def get_preferred_vehicle(runtime: ToolRuntime[Preferences]) -> str:
    """Get the preferred vehicle"""
    return runtime.context.vehicle

@tool
def get_preferred_city(runtime: ToolRuntime[Preferences]) -> str:
    """Get the preferred city"""
    return runtime.context.city

# Here is where we add the tools and the custom context to the agent
# (we reuse the model created above, with reasoning_effort="none")
agent = create_agent(
    model=model,
    tools=[get_preferred_vehicle, get_preferred_city],
    context_schema=Preferences
)

response = agent.invoke(
    {"messages": [HumanMessage(content="What is the preferred vehicle?")]},
    context=Preferences()
)

print(response['messages'][-1].content)

In [ ]:
from pprint import pprint

pprint(response)

## Let's explain the previous code in simple terms

Below is a **beginner-friendly, line-by-line explanation** of the code. We will explain **what each part does** and **why it exists**.

---

## 1. Importing `dataclass`

```python
from dataclasses import dataclass
```

* This imports `dataclass`, a Python helper that makes it easy to create simple classes used to store data.
* Think of a `dataclass` as a **clean container for variables**.

---

## 2. Creating a custom context (shared memory)

```python
# Here is where we create the custom context

@dataclass
class Preferences:
```

* You are defining a **custom context object** called `Preferences`.
* “Context” means **information the agent and tools can access while running**.

---

```python
    vehicle: str = "Vespa"
    city: str = "San Francisco"
```

* This context has two fields:

  * `vehicle` → default value `"Vespa"`
  * `city` → default value `"San Francisco"`
* If you don’t provide anything else, these defaults are used.
* You can think of this like a **user profile or settings object**.

---

## 3. Importing LangChain tool utilities

```python
from langchain.tools import tool, ToolRuntime
```

* `@tool` is a decorator that turns a Python function into a **tool an agent can call**.
* `ToolRuntime` is an object LangChain passes into tools so they can:

  * Access context
  * Access runtime data (like memory, config, etc.)

---

## 4. Creating a tool that reads from context

```python
# Here is where we use the custom context

@tool
def get_preferred_vehicle(runtime: ToolRuntime[Preferences]) -> str:
```

* This defines a **tool** named `get_preferred_vehicle`.
* `@tool` tells LangChain:
  👉 “The agent is allowed to call this function.”
* `runtime: ToolRuntime[Preferences]` means:

  * LangChain will pass in a `runtime` object
  * That runtime **contains a `Preferences` context**

---

```python
    """Get the preferred vehicle"""
```

* This is a docstring.
* LangChain uses this description to help the LLM understand **when to use the tool**.

---

```python
    return runtime.context.vehicle
```

* `runtime.context` is the current `Preferences` object.
* `.vehicle` reads the `vehicle` field.
* This returns `"Vespa"` (unless the context was changed).

---

## 5. Second tool: preferred city

```python
@tool
def get_preferred_city(runtime: ToolRuntime[Preferences]) -> str:
```

* Same idea as before, but for the city.
* It also receives the runtime and context.

---

```python
    """Get the preferred city"""
```

* Tool description for the LLM.

---

```python
    return runtime.context.city
```

* Reads `"San Francisco"` from the context and returns it.

---

## 6. Creating the agent

```python
# Here is where we add the tools and the custom context to the agent

agent = create_agent(
```

* This creates an **AI agent**.
* An agent can:

  * Read messages
  * Decide what tools to call
  * Produce a final answer

---

```python
    model="gpt-4o-mini",
```

* This tells the agent which language model to use.

---

```python
    tools=[get_preferred_vehicle, get_preferred_city],
```

* These are the **only tools the agent is allowed to use**.
* The agent can call them if it thinks they help answer the question.

---

```python
    context_schema=Preferences
```

* This is very important.
* It tells LangChain:

  * “This agent uses a context shaped like `Preferences`”
* LangChain now knows what fields exist in `runtime.context`.

---

## 7. Running the agent

```python
response = agent.invoke(
```

* This runs the agent.

---

```python
    {"messages": [HumanMessage(content="What is the preferred vehicle?")]},
```

* You send a **chat message** to the agent.
* The user is asking:
  👉 “What is the preferred vehicle?”

---

```python
    context=Preferences()
```

* You provide an **instance of the context**.
* Since you didn’t override anything:

  * `vehicle = "Vespa"`
  * `city = "San Francisco"`

---

## 8. Printing the final answer

```python
print(response['messages'][-1].content)
```

* `response['messages']` is the conversation history.
* `[-1]` means “the last message” (the agent’s reply).
* `.content` is the text of the reply.

---

## 9. What happens internally (simple version)

1. The agent receives the question
   **“What is the preferred vehicle?”**
2. The model sees that:

   * There is a tool named `get_preferred_vehicle`
   * That tool returns exactly what the question asks
3. The agent calls the tool
4. The tool reads `runtime.context.vehicle`
5. The agent responds with something like:

```
The preferred vehicle is Vespa.
```

---

## 10. One-sentence mental model

> **This code teaches an AI agent how to read structured user preferences (context) using tools and answer questions about them.**